# Week 35, Day 6: Forge: Week 35 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w035/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
from scipy.linalg import toeplitz
ev = np.linalg.eigvalsh(toeplitz(0.9 ** np.arange(20)))
print("1. ratio", round(ev.max() / ev.min()), "(it approaches ((1 + ρ)/(1 - ρ))² = 361 as the window grows)")
print("2.", 0.5 * 40, "bits per second")
print("3.", 0.6)

### Exercise: Part A: the fitting pipeline

The simulation, design helpers and Newton fitter are given. Complete the loop over basis sizes: build the design [Xs, Yh @ B], fit on the training bins and store the validation bits per spike in val[nb]. Then refit the best and report test scores.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2004)
T, D, H = 100000, 30, 50
lag = np.arange(D); hl = np.arange(1, H + 1)
k = -(np.exp(-lag / 3) - 0.6 * np.exp(-lag / 6)) * np.sin(lag / 3 + 0.3); k = 1.2 * k / np.linalg.norm(k)
h = -8 * np.exp(-hl / 1.5) - 1.5 * np.exp(-hl / 8) + 0.6 * np.exp(-((hl - 12) / 4) ** 2)
b0 = np.log(0.02)

def lagmat(x, L):
    M = np.zeros((len(x), L))
    for j in range(L):
        M[j:, j] = x[: len(x) - j]
    return M

def simulate(drive0, h, rg):
    drive = drive0.copy(); y = np.zeros(len(drive), int)
    for t in range(len(drive)):
        y[t] = rg.poisson(np.exp(min(drive[t], 3)))
        if y[t]:
            e = min(len(drive), t + 1 + len(h)); drive[t + 1:e] += y[t] * h[: e - t - 1]
    return y

def basis(nb):
    tt = np.log(hl + 2); cs = np.linspace(np.log(3), np.log(H + 2), nb); wd = cs[1] - cs[0]
    return np.array([np.where(np.abs(tt - c) < 2 * wd, 0.5 * (1 + np.cos(np.pi * (tt - c) / (2 * wd))), 0) for c in cs]).T

def fit_glm(X, y, iters=30):
    X = np.c_[X, np.ones(len(y))]; w = np.zeros(X.shape[1]); w[-1] = np.log(y.mean())
    for _ in range(iters):
        lam = np.exp(X @ w)
        w = w + np.linalg.solve((X * lam[:, None]).T @ X + 1e-6 * np.eye(len(w)), X.T @ (y - lam))
    return w

x = r.normal(size=T); Xs = lagmat(x, D)
y = simulate(Xs @ k + b0, h, r)
Yh = lagmat(np.r_[0, y[:-1]], H)
idx = np.arange(T); tr, va, te = idx < 60000, (idx >= 60000) & (idx < 80000), idx >= 80000
r0 = y[tr].mean()

def bps(X, w, m):
    eta = np.c_[X[m], np.ones(m.sum())] @ w
    return (np.sum(y[m] * eta - np.exp(eta)) - np.sum(y[m] * np.log(r0) - r0)) / (y[m].sum() * np.log(2))

w_lnp = fit_glm(Xs[tr], y[tr])
val = {}
for nb in (2, 4, 6, 10, 16):
    ...
best_nb = max(val, key=val.get)
B = basis(best_nb); Xg = np.c_[Xs, Yh @ B]; w_glm = fit_glm(Xg[tr], y[tr])
test_bps = {"LNP": bps(Xs, w_lnp, te), "GLM": bps(Xg, w_glm, te)}
k_fit, h_fit, b_fit = w_glm[:D], B @ w_glm[D:-1], w_glm[-1]
print(f"rate {y.sum() / (T / 1000):.1f} Hz; validation bits per spike", {n: round(float(v), 3) for n, v in val.items()})
print(f"chosen {best_nb} basis functions; test", {m: round(float(v), 3) for m, v in test_bps.items()})
plt.plot(hl, h, "k--", label="true"); plt.plot(hl, h_fit, label="fitted"); plt.xlabel("ms after a spike"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_choice():
    "the chosen basis size is the one with the best validation score"
    assert best_nb == max(val, key=val.get)

def test_gain():
    "on the test set, the chosen GLM beats the LNP by more than 0.15 bits per spike"
    assert test_bps["GLM"] - test_bps["LNP"] > 0.15

def test_filter():
    "the fitted stimulus filter correlates with the true one above 0.95"
    assert np.corrcoef(k_fit, k)[0, 1] > 0.95

test_choice()
test_gain()
test_filter()
print("All checks passed")

### Exercise: Part B: simulating the fitted models

Write fano_active(S), which sums each repeat's spikes in 20 ms windows and returns the mean variance-to-mean ratio across windows whose mean count exceeds 0.5. Then compute it and the smoothed PSTH correlation for the data and both models.

In [ ]:
Tr, R = 2000, 60
Xr = lagmat(r.normal(size=Tr), D)
sims = {
    "data": np.array([simulate(Xr @ k + b0, h, r) for _ in range(R)]),
    "GLM": np.array([simulate(Xr @ k_fit + b_fit, h_fit, r) for _ in range(R)]),
    "LNP": np.array([r.poisson(np.exp(np.c_[Xr, np.ones(Tr)] @ w_lnp)) for _ in range(R)]),
}

def fano_active(S):
    c = S.reshape(len(S), -1, 20).sum(2)
    ...

smooth = lambda p: np.convolve(p, np.ones(5) / 5, "same")
fano = {m: fano_active(S) for m, S in sims.items()}
psth_r = {m: np.corrcoef(smooth(S.mean(0)), smooth(sims["data"].mean(0)))[0, 1] for m, S in sims.items() if m != "data"}
print("Fano factor", {m: round(float(v), 2) for m, v in fano.items()})
print("PSTH correlation with data", {m: round(float(v), 3) for m, v in psth_r.items()})
fig, ax = plt.subplots(3, 1, sharex=True, figsize=(7, 5))
for a, (m, S) in zip(ax, sims.items()):
    tt, rr = np.nonzero(S[:20, :500].T)
    a.plot(tt, rr, "|", ms=3); a.set_ylabel(m)
ax[-1].set_xlabel("ms"); plt.show()

In [ ]:
import numpy as np
def test_fano():
    "the GLM's Fano factor is within 0.1 of the data's, and the LNP's is above 0.9"
    assert abs(fano["GLM"] - fano["data"]) < 0.1 and fano["LNP"] > 0.9

def test_psth():
    "the GLM's PSTH correlates with the data's PSTH better than the LNP's"
    assert psth_r["GLM"] > psth_r["LNP"]

test_fano()
test_psth()
print("All checks passed")